# 04 Disturbance and Impact

**Project:** Black Hills Mining Landscape Digital Twin — Phase I
**Territory:** He Sapa (Black Hills) — Unceded Lakota Territory

## Purpose

This notebook characterizes the physical legacy of mining in He Sapa:

1. **Watershed mine density** which drainages received the most disturbance
2. **Stream proximity** which mines are adjacent to named waterways
3. **Downstream exposure** contamination pathways toward Lakota lands

## The Whitewood Creek Story

Whitewood Creek drains the Lead-Deadwood area the most intensively
mined portion of He Sapa. Homestake Mine discharged cyanide-laced tailings
into Whitewood Creek for decades. The creek flows into the Belle Fourche,
then the Cheyenne River, which crosses the Cheyenne River Sioux Reservation.
This downstream pathway is the present condition, not history.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED,
    BLACK_HILLS_BBOX, OUTPUTS_DIR, FIGURES_DIR,
)
from src.loaders import load_black_hills_boundary
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings('ignore')
%matplotlib inline

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

mines = gpd.read_file(OUTPUTS_DIR/'he_sapa_mine_gazetteer.geojson')
bh    = load_black_hills_boundary()

streams_path    = OUTPUTS_DIR/'he_sapa_streams.geojson'
watersheds_path = OUTPUTS_DIR/'he_sapa_watersheds.geojson'
streams    = gpd.read_file(streams_path)    if streams_path.exists()    else gpd.GeoDataFrame()
watersheds = gpd.read_file(watersheds_path) if watersheds_path.exists() else gpd.GeoDataFrame()

print(f'Mines     : {len(mines):,}')
print(f'Streams   : {len(streams):,}')
print(f'Watersheds: {len(watersheds)}')

In [ ]:
# Print data sovereignty acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=['mrds', 'nhd', 'wbd'])

## Mine Density by Watershed

In [ ]:
if not watersheds.empty:
    mines_proj      = mines.to_crs(CRS_PROJECTED)
    watersheds_proj = watersheds.to_crs(CRS_PROJECTED)
    joined = gpd.sjoin(mines_proj, watersheds_proj, how='left', predicate='within')
    huc_col = next(
        (c for c in watersheds.columns if c.startswith('huc') and c[-1].isdigit()),
        None,
    )
    if huc_col:
        right_col = f'{huc_col}_right' if f'{huc_col}_right' in joined.columns else huc_col
        mine_counts = joined.groupby(right_col).size().rename('mine_count')
        watersheds = watersheds.merge(mine_counts, left_on=huc_col, right_index=True, how='left')
        watersheds['mine_count'] = watersheds['mine_count'].fillna(0).astype(int)
        ws_proj = watersheds.to_crs(CRS_PROJECTED)
        watersheds['area_km2']     = ws_proj.geometry.area / 1e6
        watersheds['mine_density'] = (watersheds['mine_count'] / watersheds['area_km2'] * 100).round(2)
        print('MINE DENSITY BY WATERSHED (mines per 100 km2)')
        for _, row in watersheds.sort_values('mine_density', ascending=False).iterrows():
            name = row.get('name', row.get(huc_col, 'Unknown'))
            bar  = chr(9608) * int(min(row['mine_density'], 50))
            print(f'  {str(name)[:30]:<30}: {row["mine_density"]:>6.1f}  {bar}')
else:
    print('Watershed data not available: run notebook 02 first.')

## Stream Proximity Analysis

In [ ]:
if not streams.empty:
    mines_proj   = mines.to_crs(CRS_PROJECTED)
    streams_proj = streams.to_crs(CRS_PROJECTED)
    streams_union = streams_proj.geometry.union_all()
    mines_proj['dist_to_stream_m'] = mines_proj.geometry.distance(streams_union)

    def stream_risk(d):
        if d <= 100:   return 'Direct (< 100m)'
        if d <= 500:   return 'High (100-500m)'
        if d <= 2000:  return 'Moderate (0.5-2km)'
        return 'Low (> 2km)'

    mines_proj['stream_proximity'] = mines_proj['dist_to_stream_m'].apply(stream_risk)
    mines['dist_to_stream_m']      = mines_proj['dist_to_stream_m'].values
    mines['stream_proximity']      = mines_proj['stream_proximity'].values

    print('MINE PROXIMITY TO STREAMS')
    prox_counts = mines_proj['stream_proximity'].value_counts()
    for category in ['Direct (< 100m)', 'High (100-500m)', 'Moderate (0.5-2km)', 'Low (> 2km)']:
        n   = prox_counts.get(category, 0)
        pct = n / len(mines_proj) * 100
        bar = chr(9608) * int(pct / 2)
        print(f'  {category:<25}: {n:>4} ({pct:>5.1f}%)  {bar}')
else:
    print('Stream data not available — run notebook 02 first.')
    mines['dist_to_stream_m'] = None
    mines['stream_proximity'] = 'Unknown'

## Disturbance Map

In [ ]:
PROX_COLORS = {
    'Direct (< 100m)':   '#C0392B',
    'High (100-500m)':   '#E67E22',
    'Moderate (0.5-2km)':'#F1C40F',
    'Low (> 2km)':       '#27AE60',
    'Unknown':            '#888888',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
if not watersheds.empty and 'mine_density' in watersheds.columns:
    watersheds.to_crs(3857).plot(
        ax=ax, column='mine_density', cmap='YlOrRd',
        legend=True, legend_kwds={'label':'Mines per 100 km2','shrink':0.7},
        alpha=0.8, zorder=1,
    )
if not streams.empty:
    streams.to_crs(3857).plot(ax=ax, color='#2471A3', linewidth=0.6, alpha=0.5, zorder=2)
bh.to_crs(3857).plot(ax=ax, facecolor='none', edgecolor='#2C3E50', linewidth=2, linestyle='--', zorder=3)
try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.3)
except Exception:
    pass
ax.set_axis_off()
ax.set_title('Mine Density by Watershed\n(mines per 100 km2)', fontsize=11, fontweight='bold')

ax = axes[1]
if not streams.empty:
    streams.to_crs(3857).plot(ax=ax, color='#2471A3', linewidth=0.8, alpha=0.6, zorder=1)
bh.to_crs(3857).plot(ax=ax, facecolor='none', edgecolor='#2C3E50', linewidth=2, linestyle='--', zorder=2)
if 'stream_proximity' in mines.columns:
    for category, color in PROX_COLORS.items():
        subset = mines[mines['stream_proximity'] == category]
        if not subset.empty:
            subset.to_crs(3857).plot(
                ax=ax, color=color, markersize=20, alpha=0.8, zorder=3, label=category,
            )
try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.3)
except Exception:
    pass
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=8, title='Distance to stream', framealpha=0.9)
ax.set_title('Mine Proximity to Streams\nContamination pathway risk', fontsize=11, fontweight='bold')

plt.suptitle('He Sapa Mining Disturbance and Watershed Impact\nUnceded Lakota Territory', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR/'04_disturbance_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## Exports

In [ ]:
mines.to_file(OUTPUTS_DIR/'he_sapa_mines_with_impact.geojson', driver='GeoJSON')
print('Exported: outputs/he_sapa_mines_with_impact.geojson')

if not watersheds.empty:
    watersheds.to_file(OUTPUTS_DIR/'he_sapa_watersheds_density.geojson', driver='GeoJSON')
    print('Exported: outputs/he_sapa_watersheds_density.geojson')

if 'stream_proximity' in mines.columns:
    n_direct = (mines['stream_proximity'] == 'Direct (< 100m)').sum()
    n_high   = (mines['stream_proximity'] == 'High (100-500m)').sum()
    print()
    print(f'KEY FINDING: {n_direct + n_high:,} mines ({(n_direct+n_high)/len(mines)*100:.1f}%) within 500m of a stream.')
    print(f'{n_direct:,} mines within 100m are effectively adjacent to a waterway.')

In [ ]:
print(generate_citations(['mrds', 'nhd', 'wbd']))